In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# To plot pdf for illustrator editting, change the config of matplotlib
plt.rcParams['pdf.fonttype'] = 42
# font will be Arial
plt.rcParams['font.family'] = 'Arial'
# Elsevier's popular figure style is to use a white background with black axes and grid lines.
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['grid.color'] = 'black'


In [ ]:
ratios = [1,5,30,50]
model_strategies = ["x1", "x1_over75calib", "x1_tra75"]
dfs =[]
for ratio in ratios:
    for ms in model_strategies:
        df = pd.read_csv(f"../ratio_{ratio}/metrics_diff_{ms}.txt")
        df["ratio"] = ratio
        df["ms"] = ms
        ####
        #ALLは不要なので、取り除く
        df = df[df["ms"] != "x1"]
        ####
        dfs.append(df)
df = pd.concat(dfs, axis=0)

In [ ]:
# Set up a grid of axes for boxplots, with one column for each unique value of "model"
# rotate x-axis labels for better readability
g = sns.FacetGrid(df, col="model", col_wrap=4, height = 4, aspect=0.6)
g.map(sns.boxplot, "ms", "cindex", "ratio", palette="colorblind")
g.set_xticklabels(["Train with All Calibrated", "Train with Over 75"],rotation=45)
g.set_xlabels("Strategy")
g.set_ylabels("C-Index")
plt.legend(title="Ratio", loc="upper right", bbox_to_anchor=(1.4, 1))
plt.savefig(f"../cindex_boxplot.pdf", transparent=True)

In [ ]:
from decimal import Decimal, Context, ROUND_HALF_UP
import pandas as pd

def format_sig_3(x):
    """個々の数値を有効数字3桁（0埋め、ハイフンマイナス）にする関数"""
    if pd.isna(x):
        return ""
    if x == 0:
        return "0.00" # 有効数字3桁の0
        
    is_negative = x < 0
    abs_x = abs(x)
    
    # 有効数字3桁で厳密に四捨五入
    d = Decimal(str(abs_x))
    ctx = Context(prec=3, rounding=ROUND_HALF_UP)
    normalized = ctx.create_decimal(d)
    
    s = f"{normalized:f}"
    
    # 整数部が3桁を超える場合（例: 12345 -> 12300）
    _, _, exponent = normalized.as_tuple()
    if exponent >= 0:
        s = str(int(normalized))
    
    # マイナスは通常のハイフン（-）を付与
    return f"-{s}" if is_negative else s



In [ ]:
df_temp = df.copy()
# df_temp = df_temp[["model", "ms", "ratio", "IBS"]].rename(columns={"IBS": "value"})
df_temp = df_temp[["model", "ms", "ratio", "cindex"]].rename(columns={"cindex": "value"})
summary = (
    df_temp.groupby(["model", "ms", "ratio"])
      .agg(
          Median=("value", "median"),
          Q1=("value", lambda x: x.quantile(0.25)),
          Q3=("value", lambda x: x.quantile(0.75)),
          Lower95=("value", lambda x: x.quantile(0.025)),
          Upper95=("value", lambda x: x.quantile(0.975))
      )
      .reset_index()
)
# DataFrameに適用

def make_IQR_string(row):
    """2つの列を結んで en dash の範囲文字列を作る関数"""
    min_str = format_sig_3(row["Q1"])
    max_str = format_sig_3(row["Q3"])
    
    # 範囲の区切りに en dash（–）を使用
    return f"{min_str} – {max_str}"

def make_95CI_string(row):
    """2つの列を結んで en dash の範囲文字列を作る関数"""
    min_str = format_sig_3(row["Lower95"])
    max_str = format_sig_3(row["Upper95"])
    
    # 範囲の区切りに en dash（–）を使用
    return f"{min_str} – {max_str}"


summary["IQR"] = summary.apply(make_IQR_string, axis=1)
summary["95% bootstrap CI"] = summary.apply(make_95CI_string, axis=1)
# summary.to_csv(f"../IBSS_summary.csv", index=False)
summary.to_csv(f"../cindex_summary.csv", index=False)


In [ ]:
# Set up a grid of axes for boxplots, with one column for each unique value of "model"
# rotate x-axis labels for better readability
g = sns.FacetGrid(df, col="model", col_wrap=4, height = 4, aspect=0.6)
g.map(sns.boxplot, "ms", "IBS", "ratio", palette="colorblind")
g.set_xticklabels(["Train with All Calibrated", "Train with Over 75"],rotation=45)
g.set_xlabels("Strategy")


g.set_ylabels("IBSS (compared to KM)")
plt.legend(title="Ratio", loc="upper right", bbox_to_anchor=(1.4, 1))
plt.savefig(f"../ibss_boxplot.pdf", transparent=True)

In [ ]:
# Set up a grid of axes for boxplots, with one column for each unique value of "model"
# rotate x-axis labels for better readability
g = sns.FacetGrid(df, col="model", col_wrap=4, height = 4, aspect=0.6, ylim=[-0.27,0.2])
g.map(sns.boxplot, "ms", "IBS", "ratio", palette="colorblind")
g.set_xticklabels(["Train with All Calibrated", "Train with Over 75"],rotation=45)
g.set_xlabels("Strategy")
g.set_ylabels("IBSS (compared to KM)")
plt.legend(title="Ratio", loc="lower right", bbox_to_anchor=(1.4, 1))
plt.savefig(f"../ibss_boxplot_zoom.pdf", transparent=True)

In [ ]:
df_temp = df.copy()
df_temp = df_temp[["model", "ms", "ratio", "IBS"]].rename(columns={"IBS": "value"})
summary = (
    df_temp.groupby(["model", "ms", "ratio"])
      .agg(
          Median=("value", "median"),
          Q1=("value", lambda x: x.quantile(0.25)),
          Q3=("value", lambda x: x.quantile(0.75)),
          Lower95=("value", lambda x: x.quantile(0.025)),
          Upper95=("value", lambda x: x.quantile(0.975))
      )
      .reset_index()
)

summary["IQR"] = (
    summary["Q1"].round(3).astype(str)
    + "-"
    + summary["Q3"].round(3).astype(str)
)

summary["95% bootstrap CI"] = (
    summary["Lower95"].round(3).astype(str)
    + "-"
    + summary["Upper95"].round(3).astype(str)
)
summary.to_csv(f"../ibss_summary.csv", index=False)

In [ ]:
import pickle
features = ["age", "bmi", "ALT", "AST", "SBP", "e_gfr"]
# all_results = {
# "RSF": [],
# "RSF_ISR": [],
# }
colors = {
"RSF_train_all": "#195C8C",
"RSF_train_75": "#779cb0",
"ISR_train_all": "#a22020",
"ISR_train_75": "#ad745f"
}
def mapper(k,ms):
    if k == "RSF" and ms == "x1": return print("All age results are the same as RSF, so we will not plot them"); return None
    elif k == "RSF" and ms == "x1_tra75": return "RSF_train_75"
    elif k == "RSF" and ms == "x1_over75calib": return "RSF_train_all"
    elif k == "RSF_ISR" and ms == "x1": print("All age results are the same as RSF, so we will not plot them"); return None
    elif k == "RSF_ISR" and ms == "x1_tra75": return "ISR_train_75"
    elif k == "RSF_ISR" and ms == "x1_over75calib": return "ISR_train_all"


for ratio in ratios:
    i = 0
    for feature in features:
        ps =[]
        fig, ax = plt.subplots(figsize=(6, 4))
        fig.subplots_adjust(left=0.20, right=0.95, bottom=0.20, top=0.90)
        i+=1
        for ms in model_strategies:
            if ms == "x1": continue#grid_values = pickle.load(open(f"../{feature}_grid.pkl", "rb"))
            else: grid_values = pickle.load(open(f"../{feature}_grid_over75.pkl", "rb"))
            
            pickle_path = f"../ratio_{ratio}/{ms}_{feature}_pdp.pkl"
            with open(pickle_path, "rb") as f:
                all_results= pickle.load(f) 


            summary = {}

            for k in all_results:

                summary[k] = {
                    "mean": all_results[k].mean(axis=0),
                    "lower": np.percentile(all_results[k], 2.5, axis=0),
                    "upper": np.percentile(all_results[k], 97.5, axis=0)
                }

            #fig, ax = plt.subplots(figsize=(6, 4))
            for k in summary:
                p, = ax.plot(grid_values, summary[k]["mean"], label=mapper(k,ms), color=colors[mapper(k,ms)])
                ps.append(p)

                ax.fill_between(
                    grid_values,
                    summary[k]["lower"],
                    summary[k]["upper"],
                    alpha=0.3,
                    color=colors[mapper(k,ms)]
                )


            #plt.title(f"PDP with CI: {k}")
            ax.set_xlabel(feature, fontsize=14)
            #ax.set_ylabel(f"LR_{ratio}_{ms}_risk")
            ax.set_ylabel(f"Risk", fontsize=14)
            x_ticks = ax.get_xticks()
            y_ticks = ax.get_yticks()
            ax.tick_params(axis='both', labelsize=14)
            if i==1:ax.legend(title="Model",bbox_to_anchor=(1.5, 1), loc="upper right", handles=ps, fontsize=14)
            # transparant background for illustrator editting
            ax.patch.set_alpha(0)
            #fig.savefig(f"../ratio_{ratio}/{ms}_{feature}_pdp2.pdf")
            fig.savefig(f"../ratio_{ratio}/{feature}_pdp2.pdf", transparent=True)
            